In [ ]:
# TODO
# Make notebook runable
# Update paths
# Update input data
# Delete IFFs and nurse estimates

In [ ]:
import pandas as pd
import pickle
%matplotlib inline

#Import style guide to make plots
import sys
sys.path.append('../../../../tjn_tools/')

# from style_guide import *
from tjn_tools.style_guide import *
#Import some data_processing tools to convert names to ISO3 
from tjn_tools.data_processing import *
import tjn_tools

import seaborn as sns
import numpy as np
pd.options.mode.chained_assignment = None  # default='warn'

import pylab as plt
import matplotlib as mpl
%matplotlib inline


#Avoid overlapping text|
# from adjustText import adjust_text

from IPython.core.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))

Note: Originally this was named 98.combine_parts.ipynb, but I renamed it to 98.combine_parts.ipynb to make it run last.

In [ ]:
path_files = "../../data/raw/estimations/"
path_files_final = "../../data/final/estimations/"
path_files_temp = "../../data/intermediate/estimations/"
path_figures = "../../data/final/estimations/figures/"

year_cbcr = 2018

In [ ]:
#Files from the data portal
country_level = f"{tjn_tools.paths.final_data}/20210810_country-level-data.csv"
country_year_level = f"{tjn_tools.paths.final_data}/20210810_country-year-level-data.csv"
bilateral_level = f"{tjn_tools.paths.final_data}/20210810_bilateral-year-level-data.csv"

#File ufrom the IFF analysis (202004-Risk-based%20IFF/Scripts/Analysis_2021_04.ipynb)
iff_file_path = f"{path_files_final}iff_sotj_table.csv"

#Salary nurses OECD
nurses_file = f"{path_files}oecd_nurses_clean.xlsx"

#Not sure why relative paths don't work for this file
tax_evasion_file_output = "~/Tax Justice Network Ltd/TJN - Shared Documents/Workstreams/Scale of Tax Injustice/State of Tax Justice report/2021 Report/Offshore wealth/Price of offshore full results.xlsx"

In [ ]:
info_expenditures_output = f"{path_files_final}{year_cbcr}_info_expenditures.csv"
cols_other_info = ['who_gvt_health_expenditure','Govt_exp_educ_gdp_wb',
        'total_taxes_revenue', 'cit_revenue', 'iso3', 'GDP_int', 'POP_int',
        'region_tjn',"UKt","OECD","OECD_OCT","G20","EU28","month_wage","FSI2020_Rank","FSI2020_Share","FSI2020_Score","CTHI21_Rank","CTHI21_Share","CTHI21_Score"]


In [ ]:
etr_output = f"{path_files_final}{year_cbcr}_cbcr_etr_rates.xlsx"
df_etrs = pd.read_excel(etr_output,index_col=0)
iso3_to_etr = df_etrs["ETR_total"].to_dict()

In [ ]:
# Input
file_output = f"{year_cbcr}_tax_avoidance_sotj_table.xlsx"
sotj_table_output = f"{path_files_final}{file_output}"

In [ ]:
iso3_to_cit_output = f"{path_files_final}{year_cbcr}_iso3_to_cit.dump"
iso3_to_cit = pickle.load(open(iso3_to_cit_output,"rb+"))


In [ ]:
# Output
file_output = f"{year_cbcr}combined_output.xlsx"
workstream_path = f"{tjn_tools.paths.sharepoint_root}/TJN - Shared Documents/Workstreams/Scale of Tax Injustice/State of Tax Justice report/2021 Report/Combined_offshore_corporate/"
final_table_output = f"{path_files_final}{file_output}"
final_table_output_workstream = f"{workstream_path}{file_output}"

# 1. Read data

In [ ]:
#Some variables were not added before: "EU28 OECT", "EU27", "EU27 OCT", "GBR OCT", th_eu_blacklist_201006, th_eu_greylist_201006, th_unctad2015
countryYearBase = pd.read_stata(f"{tjn_tools.paths.sharepoint_root}/TJN - Shared Documents/Workstreams/Financial Secrecy/CTHI/CTHI-2021/GSW/210211 countryYearBase for CTHI2021.dta")
countryYearBase = countryYearBase.loc[:,["country","EU27","EU27_OCT","EU28_OCT","GBR_OCT","th_eu_blacklist_201006", "th_eu_greylist_201006", "th_unctad2015"]]
countryYearBase["iso3"] = countryYearBase["country"].apply(get_iso3, print_failure = False)
countryYearBase = countryYearBase.loc[countryYearBase["country"] != "West Bank and Gaza"]
countryYearBase = countryYearBase.drop(columns=["country"])
countryYearBase = countryYearBase.drop_duplicates()
countryYearBase["iso3"].value_counts().head(2)

In [ ]:
#Class and other info file
other_info = pd.read_csv(info_expenditures_output, sep="\t", usecols=cols_other_info)
other_info = pd.merge(other_info,countryYearBase,how="left")
other_info.head()

In [ ]:
#Other estimates of tax avoidance
cols = {"iso3":"iso3","ps_sr_cobham2018":"TA: C&J short-run 2018 (USD million)","ps_lr_cobham2018":"TA: C&J long-run 2018 (USD million)","ps_torslov2020_2016": "TA: TWZ 2016 (USD million)","ps_torslov2020_2017": "TA: TWZ 2017 (USD million)","ps_jansky2019": "TA: JP 2019 (USD million)"}
ext_estimates = pd.read_csv(country_level,skiprows=1,sep="\t",usecols=(list(cols.keys())))
# cols = [_ for _ in ohter_m.columns if ("ps_" in _) and not ("_so") in _ and (_ not in ("ps_cobham2018","ps_torslov2018") )]
ext_estimates = ext_estimates[cols].dropna(thresh=2).drop_duplicates(subset=["iso3"])
ext_estimates = ext_estimates.rename(columns=cols)
ext_estimates[list(ext_estimates.columns )[1:]] /= 1E6
ext_estimates.loc[ext_estimates["iso3"]=="IND"]

In [ ]:
other_info = pd.merge(other_info,ext_estimates,how="left",validate="1:1")
other_info.head()

In [ ]:
#Income class
income_class = pd.read_csv(country_year_level,skiprows=1,sep="\t",usecols=["iso3","year","IncomeClass","GDP_int","POP_int"])#
income_class["GDPpc"] = income_class["GDP_int"]/income_class["POP_int"]

income_class["IncomeClassInt"] = pd.cut(income_class["GDPpc"],[0,1046,4096,12696,np.inf],labels=["L","LM","UM","H"])

missing_income = set(income_class["iso3"]) - set(income_class.dropna(subset=["IncomeClass"])["iso3"])
print(missing_income)

info_on_income_class = income_class.loc[~income_class["iso3"].isin(missing_income)].dropna(subset=["IncomeClass"]).drop_duplicates(subset=["iso3"],keep="last")
no_info_on_income_class = income_class.loc[income_class["iso3"].isin(missing_income)].dropna(subset=["IncomeClassInt"]).drop_duplicates(subset=["iso3"],keep="last")
no_info_on_income_class = no_info_on_income_class.rename(columns = {"IncomeClass": "IncomeClass_na","IncomeClassInt": "IncomeClass"})

income_class = pd.concat([info_on_income_class, no_info_on_income_class])
income_class["IncomeClass"] = income_class["IncomeClass"].map({'H':"High income", 'L':"Low income", 'LM':"Lower-middle income", 'UM':"Upper-middle income"})

income_class = income_class.loc[:,["iso3","IncomeClass"]]

other_info = pd.merge(other_info,income_class,how="left",validate="1:1")
other_info.head()

In [ ]:
#Data on nurses salaries
oecd_nurses = pd.read_excel(nurses_file,
              skiprows=4,na_values=[".."],
              usecols=["Year","2010","2011","2012","2013","2014","2015","2016","2017","2018","2019"])

oecd_nurses = pd.melt(oecd_nurses,id_vars="Year",value_vars=["2010","2011","2012","2013","2014","2015","2016","2017","2018","2019"],var_name="date",value_name="month_wage_nurse")
oecd_nurses = oecd_nurses.dropna()
oecd_nurses = oecd_nurses.sort_values(by=["Year","date"],ascending=False).drop_duplicates(subset=["Year"])
oecd_nurses["month_wage_nurse"] /= 12 #Make it monthy

#Change columns
oecd_nurses["iso3"] = oecd_nurses["Year"].apply(get_iso3)
oecd_nurses = oecd_nurses.dropna().drop(columns=["Year","date"])
other_info = pd.merge(other_info,oecd_nurses,how="left")

#Replace with OECD data when available
other_info.loc[~np.isnan(other_info["month_wage_nurse"]),"month_wage"] = other_info.loc[~np.isnan(other_info["month_wage_nurse"]),"month_wage_nurse"]
other_info.head()

In [ ]:
# TODO delete this section and the nurses one
# #IFF file
# iff_file = pd.read_csv(iff_file_path,sep=",")#,header=None)
# iff_file["ISO-3 code of country"] = iff_file["Name"].apply(get_iso3, print_failure=False)
# iff_file = iff_file.dropna(subset=["ISO-3 code of country"])
# del iff_file["Name"]
# iff_file.head()

In [ ]:

# x = tax_avoidance_file.dropna(subset=["MNCs"])
# x = x.loc[x["Revenue loss using CIT (M)"]>0]
# x = x.groupby("ISO-3 code of country").sum()["Revenue loss using CIT (M)"]
# y = df_merged.loc[df_merged["TA: Tax revenue loss using CIT (USD million)"]>0].groupby("ISO-3 code of country").sum()["TA: Tax revenue loss using CIT (USD million)"]

# z = pd.concat([x,y],axis=1)
# z.loc[z["Revenue loss using CIT (M)"].round(0) != z["TA: Tax revenue loss using CIT (USD million)"].round(0)].sort_values(by="Revenue loss using CIT (M)")

# tax_avoidance_file.loc[tax_avoidance_file["ISO-3 code of country"]=="TCD","CIT"]

In [ ]:
#Tax avoidance
tax_avoidance_file = pd.read_excel(sotj_table_output).drop_duplicates()
tax_avoidance_file["CIT"] /= 100
tax_avoidance_file["ETR"] /= 100
tax_avoidance_file["ISO-3 code of country"] = tax_avoidance_file["Name"].map(get_iso3).replace("SCG","SRB") #wrong in the first file

tax_avoidance_file.head()

In [ ]:
#Tax evasion
tax_evasion_file = pd.read_excel(tax_evasion_file_output)
tax_evasion_file["ISO-3 code of country"] = tax_evasion_file["Country"].map(get_iso3)
tax_evasion_file = tax_evasion_file.drop(columns=["Country"])
tax_evasion_file.head()


In [ ]:
#other_info.loc[other_info["iso3"].isin([get_iso3(_) for _ in ["Malaysia","Vietnam","Thailand","Cambodia","Indonesia","Myanmar","Philippines"]]),["iso3","month_wage"]]

In [ ]:
1977.196777*1000*0.05

In [ ]:
ta_dom = tax_avoidance_file.loc[tax_avoidance_file["MNCs"]=="Domestic",["ISO-3 code of country","Profit loss (M)",'Min. Profit loss (M)', 'Max. Profit loss (M)',"N_reporters"]]   
ta_dom.columns = list(ta_dom.columns[:1]) + [_ + " - dom" for _ in ta_dom.columns[1:]]
ta_fo = tax_avoidance_file.loc[tax_avoidance_file["MNCs"]=="Foreign (out)",["ISO-3 code of country","ETR","Profit loss (M)",'Min. Profit loss (M)', 'Max. Profit loss (M)',"Revenue loss using CIT (M)","Revenue loss using ETR (M)","Robust","N_reporters"]]
ta_fo.columns = list(ta_fo.columns[:2]) + [_ + " - for lose" for _ in ta_fo.columns[2:]]
ta_ga = tax_avoidance_file.loc[tax_avoidance_file["MNCs"]=="Foreign (in)",["ISO-3 code of country","CIT","Profit loss (M)",'Min. Profit loss (M)', 'Max. Profit loss (M)',"Revenue loss using CIT (M)","Revenue loss using ETR (M)","Robust"]]
ta_ga.columns = list(ta_ga.columns[:2]) + [_ + " - for gain" for _ in ta_ga.columns[2:]]
       
ta = pd.concat([ta_dom.set_index("ISO-3 code of country"),ta_fo.set_index("ISO-3 code of country"),ta_ga.set_index("ISO-3 code of country")],axis=1,sort=False)
ta = ta.reset_index().rename(columns={"index":"ISO-3 code of country"})

for tax in ["CIT","ETR"]:
    for var in ["Profit loss (M)",'Min. Profit loss (M)', 'Max. Profit loss (M)']:
        for end in ["- dom","- for lose","- for gain"]:
            ta[f"Revenue loss using {tax} (M) {end}"] = ta[f"{var} {end}"]*ta[tax]

ta = ta.rename(columns={"N_reporters - dom":"Reporting country"})
ta.head()

In [ ]:
def return_gain(common_var = "Profit loss (M) - ",vars_=["dom","for lose","for gain"],sign=-1):
    vals = ((ta["{}{}".format(common_var,vars_[0])]+sign*ta["{}{}".format(common_var,vars_[0])].abs())/2).fillna(0)
    print(vals)
    for v in vars_[1:]:
        vals += ((ta["{}{}".format(common_var,v)]+sign*ta["{}{}".format(common_var,v)].abs())/2).fillna(0)
    print(vals)
    return vals

for v in ["dom","for lose","for gain"]:
    for st in ["","Min. ","Max. "]:
        ta[f"{st}Profit loss (M) - {v}"] = ta[f"{st}Profit loss (M) - {v}"].fillna(0)
        
ta["ETR"] = ta["ISO-3 code of country"].map(iso3_to_etr)
ta["CIT"] = ta["ISO-3 code of country"].map(iso3_to_cit)
for st in ["","Min. ","Max. "]:
    ta[f"{st}Profit gain (M)"] = -return_gain(common_var = f"{st}Profit loss (M) - ",vars_=["dom","for lose","for gain"],sign=-1)
    ta[f"{st}Profit loss (M)"] = return_gain(common_var = f"{st}Profit loss (M) - ",vars_=["dom","for lose","for gain"],sign=1)

for st in ["","Min. ","Max. "]:
    ta[f"{st}Revenue gain using CIT (M)"] = ta["CIT"]*ta[f"{st}Profit gain (M)"]
    ta[f"{st}Revenue gain using ETR (M)"] = ta["ETR"]*ta[f"{st}Profit gain (M)"]
    ta[f"{st}Revenue loss using CIT (M)"] = ta["CIT"]*ta[f"{st}Profit loss (M)"]
    ta[f"{st}Revenue loss using ETR (M)"] = ta["ETR"]*ta[f"{st}Profit loss (M)"]

ta["Robust"] = ta["Reporting country"]+((ta["N_reporters - for lose"])>3).astype(int)

ta.sort_values(by="Profit gain (M)").tail(20)

In [ ]:
ta.loc[ta["ISO-3 code of country"]=="USA",[_ for _ in ta.columns if ("Revenue loss" in _) and ("." not in _) and ("CIT" in _)]]

In [ ]:
ta.loc[ta["ISO-3 code of country"]=="TCD",[_ for _ in ta.columns if ("Revenue loss" in _) and ("." not in _) and ("CIT" in _)]]

In [ ]:
ta.loc[ta["ISO-3 code of country"]=="TCD",[_ for _ in ta.columns if ("Profit loss" in _) and ("." not in _) ]]

In [ ]:
## MERGE FILES
df_merged = pd.merge(tax_evasion_file,ta,how="outer").dropna(subset=["ISO-3 code of country"])
# df_merged = pd.merge(df_merged,iff_file,how="outer").dropna(subset=["ISO-3 code of country"]) #TODO delete this line and nurses-wise one
# df_merged = pd.merge(df_merged,childrens_file,how="left").dropna(subset=["ISO-3 code of country"])
df_merged = pd.merge(df_merged,other_info,left_on="ISO-3 code of country",right_on="iso3",how="left")

df_merged["Offshore wealth owned by citizens of country (USD billion)"] *= 1000*0.05 #Convert to million and multiply by the rate of return
df_merged["FSI2020_Share"] *= 100
df_merged["CTHI21_Share"] *= 100

df_merged["CIT"] = df_merged["ISO-3 code of country"].map(iso3_to_cit)
df_merged.head()


In [ ]:
tax_avoidance_file.loc[tax_avoidance_file["ISO-3 code of country"]=="TCD"]

In [ ]:
rename_cols = {}
for st in ["","Min. ","Max. "]:
    rename_cols[f"{st}Profit gain (M)"] = f"TA: {st}Tax base gain (USD million)"
    rename_cols[f"{st}Profit loss (M)"] = f"TA: {st}Tax base loss (USD million)"
    rename_cols[f"{st}Revenue gain using CIT (M)"] = f"TA: {st}Tax revenue gain using CIT (USD million)"
    rename_cols[f"{st}Revenue gain using ETR (M)"] = f"TA: {st}Tax revenue gain using ETR (USD million)"
    rename_cols[f"{st}Revenue loss using CIT (M)"] = f"TA: {st}Tax revenue loss using CIT (USD million)"
    rename_cols[f"{st}Revenue loss using ETR (M)"] = f"TA: {st}Tax revenue loss using ETR (USD million)"
    

rename_cols.update({"Offshore wealth owned by citizens of country (USD billion)": 'OW: Tax base loss (USD million)',
 "Tax revenue loss: Offshore wealth (USD million)": 'OW: Tax revenue loss (USD million)',
 'Share of global tax loss inflicted by country': 'Harm OW: Total (% total)',
 'Tax loss inflicted on other countries': 'Harm OW: Total (USD million)',
 'Reporting country': 'TA: Reporting country',
 'Robust': 'TA: Robust',
 'N_reporters - for lose': 'TA: Number countries reporting',
 'Outward Banking Positions': 'IFF: Outward Banking Positions',
 'Inward Banking Positions': 'IFF: Inward Banking Positions',
 'Outward FDI': 'IFF: Outward FDI',
 'Inward FDI': 'IFF: Inward FDI',
 'Outward Portfolio Inv.': 'IFF: Outward Portfolio Inv.',
 'Inward Portfolio Inv.': 'IFF: Inward Portfolio Inv.',
 'Outward Trade (Exports)': 'IFF: Outward Trade (Exports)',
 'Inward Trade (Imports)': 'IFF: Inward Trade (Imports)',
 'Top Flow': 'IFF: Top Flow',
 'Top Vulnerability': 'IFF: Top Vulnerability',
 'Vulnerability Region': 'IFF: Vulnerability Region',
 'Top1': 'IFF: Top1',
 'Top2': 'IFF: Top2',
 'Top3': 'IFF: Top3',
 'FSI2020_Rank': 'FSI_Rank',
 'FSI2020_Share': 'FSI_Share',
 'FSI2020_Score': 'FSI_Score',
 'CTHI21_Rank': 'CTHI_Rank',
 'CTHI21_Share': 'CTHI_Share',
 'CTHI21_Score': 'CTHI_Score',
 #'Children lives lost due to tax revenue loss (total)': 'Children lives lost due to tax revenue loss (total)',
 'Govt_exp_educ_gdp_wb': 'WBD: Government education expenditure',
 'who_gvt_health_expenditure': 'WHO: Government health expenditure',
 'total_taxes_revenue': 'GRD: Total tax revenue',
 'cit_revenue': 'GRD: Total corporate income revenue',
 'GDP_int': 'GDP',
 'POP_int': 'POP',
 'region_tjn': 'Region',
 'IncomeClass': 'Income Class',
 'UKt': 'UK territory',
 'month_wage': 'Average wage'})

In [ ]:
tjn_tools.iso3_to_name("ESP")

In [ ]:
df_merged = df_merged.rename(columns = rename_cols)
df_merged = df_merged.dropna(subset=["OW: Tax base loss (USD million)","Harm OW: Total (USD million)","TA: Tax base loss (USD million)","TA: Tax base gain (USD million)"],how="all")
df_merged = df_merged.loc[df_merged["ISO-3 code of country"]!="ATA"]
df_merged["Country"] = df_merged["ISO-3 code of country"].map(iso3_to_name)
df_merged.loc[df_merged["ISO-3 code of country"]=="PUS","Region"] = 'Caribean/American isl.'
df_merged["IncomeClass2"] = df_merged["Income Class"].str.contains("Low").replace({False: "Higher", True: "Lower"})
df_merged.head()

In [ ]:
#Impute missing values
df_merged["health_gdp"] = df_merged["WHO: Government health expenditure"]/df_merged["GDP"]
df_merged["educ_gdp"] = df_merged["WBD: Government education expenditure"]/df_merged["GDP"]
df_merged["tax_rev_gdp"] = df_merged["GRD: Total tax revenue"]/df_merged["GDP"]
df_merged["c_tax_rev_gdp"] = df_merged["GRD: Total corporate income revenue"]/df_merged["GDP"]

reg_av = df_merged.groupby("Income Class").sum()
reg_av["health_gdp"] = reg_av["WHO: Government health expenditure"]/reg_av["GDP"]
reg_av["educ_gdp"] = reg_av["WBD: Government education expenditure"]/reg_av["GDP"]
reg_av["tax_rev_gdp"] = reg_av["GRD: Total tax revenue"]/reg_av["GDP"]
reg_av["c_tax_rev_gdp"] = reg_av["GRD: Total corporate income revenue"]/reg_av["GDP"]

reg_av = reg_av.to_dict()

for v in ["health_gdp","educ_gdp","tax_rev_gdp","c_tax_rev_gdp"]:
    df_merged.loc[np.isnan(df_merged[v]),v] =  df_merged.loc[np.isnan(df_merged[v]),"Income Class"].map(reg_av[v])

df_merged["WHO: Government health expenditure (imp)"] = df_merged["health_gdp"]*df_merged["GDP"]
df_merged["WBD: Government education expenditure (imp)"] = df_merged["educ_gdp"]*df_merged["GDP"]
df_merged["GRD: Total tax revenue (imp)"] = df_merged["tax_rev_gdp"]*df_merged["GDP"]
df_merged["GRD: Total corporate income revenue (imp)"] = df_merged["c_tax_rev_gdp"]*df_merged["GDP"]

In [ ]:
#Tax losses
df_merged["Loss OW (USD million)"] = df_merged['OW: Tax revenue loss (USD million)'].fillna(0)
df_merged.loc[df_merged["Loss OW (USD million)"]<0,"Loss OW (USD million)"] = 0
for st in ["","Min. ","Max. "]:
    df_merged[f"{st}Loss TA using CIT (USD million)"] = df_merged['TA: Tax revenue loss using CIT (USD million)'].fillna(0)
    df_merged[f"{st}Loss TA using ETR (USD million)"] = df_merged['TA: Tax revenue loss using ETR (USD million)'].fillna(0)


df_merged["GDP (only lossers)"] = df_merged["GDP"].copy()
df_merged.loc[df_merged["GDP (only lossers)"]<0,"GDP (only lossers)"] = 0

for tax in ["CIT","ETR"]:
    df_merged[f"Loss Total using {tax} (USD million)"] = df_merged[f"Loss TA using {tax} (USD million)"].fillna(0) + df_merged["Loss OW (USD million)"].fillna(0)
    df_merged[f"Loss Total using {tax} (% GDP)"] = 100*1E6*df_merged[f"Loss Total using {tax} (USD million)"]/df_merged["GDP"]
    df_merged[f"Loss Total using {tax} (% gvt tax revenue)"] = 100*1E6*df_merged[f"Loss Total using {tax} (USD million)"]/df_merged["GRD: Total tax revenue"]
    df_merged[f"Loss Total using {tax} Global (% GDP)"] = 100*1E6*df_merged[f"Loss Total using {tax} (USD million)"].sum()/df_merged["GDP"].sum()
    df_merged[f"Loss Total using {tax} Regional (% GDP)"] = 100*1E6*df_merged.groupby("Region")[f"Loss Total using {tax} (USD million)"].transform(sum)/df_merged.groupby("Region")["GDP"].transform(sum)
    df_merged[f"Loss Total using {tax} Global (% gvt tax revenue)"] = 100*1E6*df_merged[f"Loss Total using {tax} (USD million)"].sum()/df_merged["GRD: Total tax revenue (imp)"].sum()
    df_merged[f"Loss Total using {tax} Regional (% gvt tax revenue)"] = 100*1E6*df_merged.groupby("Region")[f"Loss Total using {tax} (USD million)"].transform(sum)/df_merged.groupby("Region")["GRD: Total tax revenue (imp)"].transform(sum)
    df_merged[f"Loss Total using {tax} (per capita)"] = 1E6*df_merged[f"Loss Total using {tax} (USD million)"]/df_merged["POP"]
    df_merged[f"Loss Total using {tax} (% Education)"] = 100*1E6*df_merged[f"Loss Total using {tax} (USD million)"]/df_merged['WBD: Government education expenditure']
    df_merged[f"Loss Total using {tax} (% Health)"] = 100*1E6*df_merged[f"Loss Total using {tax} (USD million)"]/df_merged["WHO: Government health expenditure"]
    df_merged[f"Loss Total using {tax} (%  gvt corporate revenue)"] = 100*1E6*df_merged[f"Loss Total using {tax} (USD million)"]/df_merged["GRD: Total corporate income revenue"]
    df_merged[f"Loss Total using {tax} (%  gvt tax revenue)"] = 100*1E6*df_merged[f"Loss Total using {tax} (USD million)"]/df_merged["GRD: Total tax revenue"]
    df_merged[f"Loss Total using {tax} (# Nurses)"] = 1E6*df_merged[f"Loss Total using {tax} (USD million)"]/df_merged["Average wage"]/12


In [ ]:
df_merged.loc[df_merged["ISO-3 code of country"]=="USA",[f"Loss Total using CIT (% gvt tax revenue)","Loss Total using CIT (USD million)","Loss TA using CIT (USD million)","Loss OW (USD million)"]]

In [ ]:
df_merged["TA: Tax revenue loss using ETR (USD million)"].sum()
print("TRL {0:2,.0f}B (95% CI {1:2,.0f}-{2:2,.0f}B)".format(*1e-3*df_merged[["TA: Tax revenue loss using CIT (USD million)","TA: Min. Tax revenue loss using CIT (USD million)","TA: Max. Tax revenue loss using CIT (USD million)"]].sum()))

In [ ]:
#Harm to others
#OW: Already in the file
df_merged["Harm OW: Total (% total)"] *= 100

for st in ["","Min. ","Max. "]:
    #Total tax lost
    #OW already in the file
    df_merged[f"{st}Base gain TA (USD million)"] = df_merged[f'TA: {st}Tax base gain (USD million)'].fillna(0)
    df_merged[f"{st}Harm TA: Total (% total)"] = 100*df_merged[f"{st}Base gain TA (USD million)"]/df_merged[f"{st}Base gain TA (USD million)"].sum()
    
    total_loss_ta_cit = df_merged[f"{st}Loss TA using CIT (USD million)"].sum()
    total_loss_ta_etr = df_merged[f"{st}Loss TA using ETR (USD million)"].sum()
    df_merged[f"{st}Harm TA: Total using CIT (USD million)"] = total_loss_ta_cit*df_merged[f"{st}Harm TA: Total (% total)"].fillna(0)/100
    df_merged[f"{st}Harm TA: Total using ETR (USD million)"] = total_loss_ta_etr*df_merged[f"{st}Harm TA: Total (% total)"].fillna(0)/100

for tax in ["CIT","ETR"]:
    total_loss = df_merged[f"Loss Total using {tax} (USD million)"].sum()
    df_merged[f"Harm: Total using {tax} (USD million)"] = df_merged[f"Harm TA: Total using {tax} (USD million)"] + df_merged["Harm OW: Total (USD million)"]
    df_merged[f"Harm: Total using {tax} (% total)"] = 100*df_merged[f"Harm: Total using {tax} (USD million)"]/(df_merged[f"Harm TA: Total using {tax} (USD million)"] + df_merged["Harm OW: Total (USD million)"]).sum()

    total_nurses = df_merged[f'Loss Total using {tax} (# Nurses)'].sum()
    df_merged[f"Harm: Total using {tax} (# Nurses)"] = total_nurses*df_merged[f"Harm: Total using {tax} (% total)"]/100

df_merged["Year"] = 2021

#Add missing values
df_merged.loc[np.isnan(df_merged["OW: Tax base loss (USD million)"]),
              ['OW: Tax base loss (USD million)',
 'OW: Tax revenue loss (USD million)',
 'Harm OW: Total (% total)',
 'Harm OW: Total (USD million)','Loss OW (USD million)']] = np.NaN

cond = np.isnan(df_merged["TA: Tax base loss (USD million)"]) & np.isnan(df_merged["OW: Tax base loss (USD million)"])
df_merged.loc[cond,['Loss Total (USD million)',
 'Loss Total (% GDP)',
 'Loss Total (% gvt tax revenue)',
 'Loss Total Global (% GDP)',
 'Loss Total Regional (% GDP)',
 'Loss Total Global (% gvt tax revenue)',
 'Loss Total Regional (% gvt tax revenue)',
 'Loss Total (per capita)',
 'Loss Total (% Education)',
 'Loss Total (% Health)',
 'Loss Total (%  gvt corporate revenue)',
 'Loss Total (%  gvt tax revenue)',
 'Loss Total (# Nurses)']] = np.nan

In [ ]:

df_merged = pd.merge(df_merged,other_info,how="outer")
df_merged.loc[df_merged["Country"]=="India"]

In [ ]:
def robust(s):
    return a

a = []
for i,row in df_merged.iterrows():
    if row["TA: Robust"]==2:
        a.append("background-color: #44b0c6")
    elif row["TA: Robust"]==1:
        a.append("background-color: #94c9d4")
    else:
        a.append("background-color: white")
        
    
# df_merged.style.apply(robust)

In [ ]:
df_merged.style.apply(robust).to_excel("~/Downloads/combined_output.xlsx",index=None)
df_merged.style.apply(robust).to_excel(final_table_output,index=None)
df_merged.style.apply(robust).to_excel(final_table_output_workstream,index=None)

In [ ]:
df_merged.loc[df_merged["Income Class"].fillna("X")=="X"]

In [ ]:
# ## Work for ATP (danish people)
# #Main results
# final_data_output = f"{path_files_temp}{year_cbcr}_replicates.csv"
# atp_prbook = pd.read_csv(final_data_output,sep="\t").reset_index(drop=True)
# atp_prbook = atp_prbook.dropna()
# atp_prbook = atp_prbook.loc[atp_prbook["profits"]>0]
# atp_prbook = atp_prbook.groupby(["iso3_d","n_rep"]).sum()[["profits"]].groupby("iso3_d").median().reset_index()
# atp_prbook.head()



In [ ]:
# atp_other = pd.read_excel(final_table_output_workstream)
# atp_other["Gain - Loss"] = atp_other["TA: Tax base gain (USD million)"] -  atp_other["TA: Tax base loss (USD million)"]
# atp_other = atp_other[['ISO-3 code of country','ETR', "Gain - Loss"]]
# atp_other.columns = ["iso3_d","ETR","Gain - Loss"]
# atp_other = atp_other.loc[atp_other["Gain - Loss"]>0]
# atp_other = pd.merge(atp_other,atp_prbook,how="left")
# atp_other["profits"] /= 1E6

In [ ]:
# atp_other["var"] = 100*atp_other["Gain - Loss"]/atp_other["profits"] * atp_other["Gain - Loss"]/atp_other["Gain - Loss"].sum()
# atp_other=  atp_other.sort_values(by="var",ascending=False)
# atp_other.to_excel("C:/Users/javga/Downloads/temp.xlsx")

In [ ]:
# atp_other.loc[atp_other["profits"]<atp_other["Gain - Loss"]]